# DART 재무데이터 수집·변환 품질 점검 **v1.1**

**v1.0 → v1.1 (SK하이닉스 CF 덤프 기반 수정):**
1. **분류체계 전환 대응** — `ifrs_`(~2019) / `ifrs-full_`(2019~)로 쪼개진 계정을
   분기별 우선순위 선택(per-period pick)으로 병합. 이전 버전은 2015~2018 히스토리가 잘렸음.
2. **BS proxy 필드 추가** — `ppe`(유형자산), `intangible_assets`(무형자산)
3. **D&A proxy 셀 신설** — CF 본문에 감가상각비를 안 싣는 기업(SK하이닉스 등) 대응:
   `D&A_q ≈ CapEx_q − Δ(유형+무형 장부가)`. 실측 보유 종목(삼성전자)으로 정확도 교차검증.

- Cell 1 : 설정 / Cell 2 : 변환 모듈 / Cell 3 : 매핑 진단
- Cell 4 : 품질 점검 실행 / Cell 5 : D&A proxy 검증

In [7]:
# ═══════════════════════════════════════════════════════════════
#  ★ 입력 변수 — 이 셀만 수정하세요
# ═══════════════════════════════════════════════════════════════
# 점검 대상 (6자리 코드, 'A' 접두어 허용)
CHECK_TICKERS = [
    "005930",   # 삼성전자
    "000660",   # SK하이닉스
    "035420",   # NAVER
    "005380",   # 현대차
    "051910",   # LG화학
]

DB_INFO = {
    "host": "192.168.0.230",
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
TABLE_DART_FS = "korea_fs_data_from_DART_V2"

VERBOSE = True
print("[OK] 설정 완료 —", CHECK_TICKERS)


[OK] 설정 완료 — ['005930', '000660', '035420', '005380', '051910']


## Cell 2 · DART → 표준 wide 변환 (v1.1)

In [8]:
# ═══════════════════════════════════════════════════════════════
#  DART long → 표준 wide 변환 모듈
#  - account_id 우선 매칭, 실패 시 account_nm 정규식 fallback
#  - IS/CIS: 누적 vs 3개월 자동 감지 후 분기화 (Q4 = FY − 3개분기)
#  - CF: 항상 누적으로 간주 → 차분 (Q2=H1−Q1, Q3=Q3−H1, Q4=FY−Q3)
#  - BS: 시점 잔액 그대로
# ═══════════════════════════════════════════════════════════════
import re
import numpy as np
import pandas as pd
import pymysql
from datetime import datetime


def log(tag, msg):
    print(f"[{datetime.now():%H:%M:%S}][{tag}] {msg}", flush=True)


def norm_ticker(t: str) -> str:
    """'A005930' → '005930'"""
    t = str(t).strip().upper()
    return t[1:].zfill(6) if t.startswith("A") else t.zfill(6)


def to_dg_ticker(t: str) -> str:
    """'005930' → 'A005930' (forecast 테이블용)"""
    return "A" + norm_ticker(t)


# ───────────────────────────────────────────────────────────────
#  표준 필드 매핑 정의
#    ids : account_id 후보 (정확 일치, 접두어 ifrs_/ifrs-full_ 모두 등록)
#    nm  : account_nm 정규식 후보 (앞에 있을수록 우선)
#    sj  : 허용 재무제표 (앞에 있을수록 우선; IS 우선, 없으면 CIS)
#    agg : 'pick'=대표 계정 1개 선택(중복합산 방지) / 'sum'=매칭 계정 전부 합산
# ───────────────────────────────────────────────────────────────
def _ids(*stems):
    out = []
    for s in stems:
        out += [f"ifrs_{s}", f"ifrs-full_{s}"]
    return out


FIELD_MAP = {
    # ── 손익 (flow) ──
    "revenue": dict(
        ids=_ids("Revenue") + ["dart_Revenue"],
        nm=[r"^매출액$", r"^매출$", r"^수익\(매출액\)$", r"^영업수익$"],
        sj=["IS", "CIS"], agg="pick"),
    "operating_income": dict(
        ids=["dart_OperatingIncomeLoss"] + _ids("OperatingIncomeLoss"),
        nm=[r"^영업이익", r"^영업손익"],
        sj=["IS", "CIS"], agg="pick"),
    "pretax_income": dict(
        ids=_ids("ProfitLossBeforeTax"),
        nm=[r"법인세비용차감전", r"^세전.*이익"],
        sj=["IS", "CIS"], agg="pick"),
    "tax_expense": dict(
        ids=_ids("IncomeTaxExpenseContinuingOperations", "IncomeTaxExpense"),
        nm=[r"^법인세비용"],
        sj=["IS", "CIS"], agg="pick"),
    "interest_expense": dict(
        ids=_ids("FinanceCosts") + ["dart_InterestExpenseFinanceExpense"],
        nm=[r"^이자비용", r"^금융비용"],
        sj=["IS", "CIS"], agg="pick"),

    # ── 현금흐름 (flow, 누적) ──
    "da_cf": dict(
        ids=["dart_AdjustmentsForDepreciationExpense"] +
            _ids("AdjustmentsForDepreciationExpense",
                 "AdjustmentsForDepreciationAndAmortisationExpense",
                 "DepreciationAndAmortisationExpense"),
        nm=[r"감가상각비와\s*무형자산상각", r"감가상각비\s*및\s*상각",
            r"감가상각"],                    # 앞머리 고정 제거 — "유형자산 감가상각비" 등 대응
        sj=["CF"], agg="pick"),
    "intangible_amort_cf": dict(
        ids=["dart_AmortisationExpense"] +
            _ids("AdjustmentsForAmortisationExpense", "AmortisationExpense"),
        nm=[r"^(?!.*감가상각).*무형자산\s*상각"],   # 합산계정(감가상각비와 무형자산상각비) 제외 — da_cf 와 이중계상 방지
        sj=["CF"], agg="pick"),
    "capex_tangible": dict(
        ids=_ids("PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities",
                 "PurchaseOfPropertyPlantAndEquipment"),
        nm=[r"유형자산의\s*취득", r"유형자산의\s*증가", r"유형자산\s*취득",
            r"토지.*취득|건설중인자산.*(?:취득|증가)"],
        sj=["CF"], agg="pick"),
    "capex_intangible": dict(
        ids=_ids("PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities",
                 "PurchaseOfIntangibleAssets"),
        nm=[r"무형자산의\s*취득", r"무형자산의\s*증가", r"무형자산\s*취득"],
        sj=["CF"], agg="pick"),

    # ── 재무상태 (stock) ──
    "receivables": dict(
        ids=_ids("TradeAndOtherCurrentReceivables", "CurrentTradeReceivables"),
        nm=[r"^매출채권$", r"^매출채권\s*및", r"^매출채권과"],
        sj=["BS"], agg="pick"),
    "inventories": dict(
        ids=_ids("Inventories"),
        nm=[r"^재고자산"],
        sj=["BS"], agg="pick"),
    "prepaid_expenses": dict(
        ids=[], nm=[r"^선급비용"], sj=["BS"], agg="pick"),
    "payables": dict(
        ids=_ids("TradeAndOtherCurrentPayables", "CurrentTradePayables"),
        nm=[r"^매입채무$", r"^매입채무\s*및", r"^매입채무와"],
        sj=["BS"], agg="pick"),
    "accrued_expenses": dict(
        ids=[], nm=[r"^미지급비용"], sj=["BS"], agg="pick"),
    "other_payables": dict(
        ids=[], nm=[r"^미지급금"], sj=["BS"], agg="pick"),
    "advances_received": dict(
        ids=[], nm=[r"^선수금"], sj=["BS"], agg="pick"),
    "contract_liabilities": dict(
        ids=_ids("ContractLiabilities"),
        nm=[r"^계약부채"], sj=["BS"], agg="pick"),

    "short_term_debt": dict(
        ids=["dart_ShortTermBorrowings"] + _ids("ShorttermBorrowings"),
        nm=[r"^단기차입금"], sj=["BS"], agg="pick"),
    "current_lt_debt": dict(
        ids=[], nm=[r"^유동성장기부채", r"^유동성장기차입금", r"^유동성사채"],
        sj=["BS"], agg="sum"),
    "bonds": dict(
        ids=[], nm=[r"^사채$", r"^사채\("], sj=["BS"], agg="pick"),
    "long_term_debt": dict(
        ids=["dart_LongTermBorrowingsGross"],
        nm=[r"^장기차입금"], sj=["BS"], agg="pick"),
    "lease_liab": dict(
        ids=_ids("LeaseLiabilities", "CurrentLeaseLiabilities",
                 "NoncurrentLeaseLiabilities"),
        nm=[r"리스부채"], sj=["BS"], agg="sum"),

    "cash": dict(
        ids=_ids("CashAndCashEquivalents"),
        nm=[r"^현금및현금성자산"], sj=["BS"], agg="pick"),
    "short_term_invest": dict(
        ids=["dart_ShortTermDepositsNotClassifiedAsCashEquivalents"],
        nm=[r"^단기금융상품", r"^단기투자자산"], sj=["BS"], agg="pick"),
    "total_equity": dict(
        ids=_ids("Equity"),
        nm=[r"^자본총계"], sj=["BS"], agg="pick"),
    "ppe": dict(
        ids=_ids("PropertyPlantAndEquipment"),
        nm=[r"^유형자산$"], sj=["BS"], agg="pick"),
    "intangible_assets": dict(
        ids=_ids("IntangibleAssetsOtherThanGoodwill", "IntangibleAssets"),
        nm=[r"^무형자산$"], sj=["BS"], agg="pick"),
    "total_assets": dict(
        ids=_ids("Assets"),
        nm=[r"^자산총계"], sj=["BS"], agg="pick"),
}

QUARTER_ORDER = {"Q1": 1, "H1": 2, "Q3": 3, "FY": 4}
FLOW_SJ  = {"IS", "CIS", "CF"}


def _load_ticker_long(ticker: str, db_info: dict, table: str) -> pd.DataFrame:
    conn = pymysql.connect(**db_info, charset="utf8mb4")
    try:
        df = pd.read_sql(
            f"""SELECT bsns_year, quarter, sj_div, account_id, account_nm,
                       thstrm_amount, report_date
                FROM {table} WHERE ticker = %s""",
            conn, params=[norm_ticker(ticker)])
    finally:
        conn.close()
    df["thstrm_amount"] = pd.to_numeric(df["thstrm_amount"], errors="coerce")
    df["report_date"] = pd.to_datetime(df["report_date"])
    return df


def _match_field(df: pd.DataFrame, spec: dict) -> pd.DataFrame:
    """
    필드 정의(spec)에 맞는 행 선택. 반환: (bsns_year, quarter) 별 단일 값.

    ★ v1.1: 기간별 선택(per-period pick)으로 변경.
      K-IFRS 분류체계가 2019년경 ifrs_ → ifrs-full_ 로 전환되어 같은 항목이
      기간별로 다른 account_id 로 쪼개져 있음. 대표 계정 1개만 고르면
      전환 이전/이후 한쪽 히스토리가 통째로 사라지므로, (연도,분기)마다
      우선순위(id 순서 > nm 패턴 순서 > 커버리지)가 가장 높은 행 1개를 선택.
      같은 분기에 유사 계정이 중복돼도 1개만 뽑아 이중계상은 여전히 차단됨.
    """
    sub = df[df["sj_div"].isin(spec["sj"])].copy()
    if sub.empty:
        return pd.DataFrame()

    # 후보 수집 + 우선순위(rank) 부여: id 일치(0~) < nm 패턴(1000~)
    id_rank = {aid: i for i, aid in enumerate(spec["ids"])}
    cand = sub[sub["account_id"].isin(id_rank)].copy()
    if not cand.empty:
        cand["rank"] = cand["account_id"].map(id_rank)
    if spec["nm"]:
        rest = sub.drop(index=cand.index) if not cand.empty else sub
        parts = [cand] if not cand.empty else []
        for j, pat in enumerate(spec["nm"]):
            m = rest[rest["account_nm"].astype(str).str.strip()
                        .str.contains(pat, regex=True, na=False)]
            if not m.empty:
                m = m.copy()
                m["rank"] = 1000 + j
                parts.append(m)
                rest = rest.drop(index=m.index)
        cand = pd.concat(parts) if parts else cand
    if cand.empty:
        return pd.DataFrame()

    # sj_div 우선순위 (IS > CIS 등): 상위 sj에 데이터가 있으면 그것만
    for sj in spec["sj"]:
        c2 = cand[cand["sj_div"] == sj]
        if not c2.empty:
            cand = c2
            break

    if spec["agg"] == "sum":
        # 같은 분기의 서로 다른 계정을 합산 (리스부채 유동+비유동 등)
        out = (cand.groupby(["bsns_year", "quarter"], as_index=False)
                   ["thstrm_amount"].sum(min_count=1))
    else:
        # per-period pick: 분기별로 rank 최소(동률이면 전체 커버리지 최대) 행 1개
        cov = cand.groupby("account_id")["bsns_year"].count()
        cand["cov"] = cand["account_id"].map(cov)
        cand = cand.sort_values(["rank", "cov"], ascending=[True, False])
        out = cand.drop_duplicates(subset=["bsns_year", "quarter"], keep="first")[
            ["bsns_year", "quarter", "thstrm_amount"]].copy()
    return out


def _detect_cumulative(pivot: pd.DataFrame) -> bool:
    """
    IS 계열 flow 가 누적인지 3개월치인지 자동 감지.
    (Q1+H1+Q3)/FY 중앙값: 3개월치 ≈ 0.75, 누적 ≈ 1.5 → 임계 1.1
    """
    ratios = []
    for y, row in pivot.iterrows():
        if all(pd.notnull(row.get(q)) for q in ("Q1", "H1", "Q3", "FY")) \
                and row["FY"] not in (0, None):
            ratios.append((row["Q1"] + row["H1"] + row["Q3"]) / row["FY"])
    if not ratios:
        return False   # 판단 불가 → 3개월치 가정 (보수적)
    return float(np.median(ratios)) > 1.1


def _flow_to_quarterly(series_df: pd.DataFrame, force_cumulative: bool = None):
    """
    flow 항목 (연도,분기,값) → 분기화 값 dict {(year,'Qn'): value}.
    force_cumulative: None=자동감지, True=누적 차분, False=3개월치 취급
    반환: (dict, cumulative여부)
    """
    pivot = series_df.pivot_table(index="bsns_year", columns="quarter",
                                  values="thstrm_amount", aggfunc="first")
    cum = _detect_cumulative(pivot) if force_cumulative is None else force_cumulative

    out = {}
    for y, row in pivot.iterrows():
        q1, h1, q3, fy = (row.get("Q1"), row.get("H1"),
                          row.get("Q3"), row.get("FY"))
        if cum:
            out[(y, "Q1")] = q1
            out[(y, "Q2")] = h1 - q1 if pd.notnull(h1) and pd.notnull(q1) else np.nan
            out[(y, "Q3")] = q3 - h1 if pd.notnull(q3) and pd.notnull(h1) else np.nan
            out[(y, "Q4")] = fy - q3 if pd.notnull(fy) and pd.notnull(q3) else np.nan
        else:
            out[(y, "Q1")] = q1
            out[(y, "Q2")] = h1
            out[(y, "Q3")] = q3
            if all(pd.notnull(v) for v in (fy, q1, h1, q3)):
                out[(y, "Q4")] = fy - (q1 + h1 + q3)
            else:
                out[(y, "Q4")] = np.nan
    return out, cum


_QDATE = {"Q1": "-03-31", "Q2": "-06-30", "Q3": "-09-30", "Q4": "-12-31"}


def load_dart_financials_wide(ticker: str, db_info: dict,
                              table_name: str = None,
                              item_keys=None, fillna_zero: bool = False,
                              verbose: bool = False) -> pd.DataFrame:
    """
    DART long 테이블 → 분기 wide DataFrame (index=분기말 date, 단위=원).
    기존 load_korea_financials_wide 와 동일한 사용 패턴.
    """
    table_name = table_name or TABLE_DART_FS
    raw = _load_ticker_long(ticker, db_info, table_name)
    if raw.empty:
        return pd.DataFrame()

    fields = item_keys or list(FIELD_MAP.keys())
    col_data, cum_info = {}, {}

    for f in fields:
        spec = FIELD_MAP[f]
        sel = _match_field(raw, spec)
        if sel.empty:
            continue
        if spec["sj"][0] in FLOW_SJ:
            force = True if spec["sj"] == ["CF"] else None   # CF는 항상 누적
            qvals, cum = _flow_to_quarterly(sel, force_cumulative=force)
            cum_info[f] = cum
        else:  # BS: 시점 잔액, H1→Q2 라벨만 변경
            qvals = {}
            for _, r in sel.iterrows():
                q = {"Q1": "Q1", "H1": "Q2", "Q3": "Q3", "FY": "Q4"}[r["quarter"]]
                qvals[(int(r["bsns_year"]), q)] = r["thstrm_amount"]
        col_data[f] = qvals

    if not col_data:
        return pd.DataFrame()

    all_keys = sorted({k for v in col_data.values() for k in v})
    idx = pd.to_datetime([f"{y}{_QDATE[q]}" for y, q in all_keys])
    wide = pd.DataFrame(
        {f: [col_data[f].get(k, np.nan) for k in all_keys] for f in col_data},
        index=idx).sort_index()

    if fillna_zero:
        wide = wide.fillna(0.0)

    if verbose:
        cum_flows = [f for f, c in cum_info.items() if c]
        log(norm_ticker(ticker),
            f"wide shape={wide.shape} 기간={wide.index.min().date()}~{wide.index.max().date()}"
            + (f"  누적차분 적용: {cum_flows}" if cum_flows else ""))
    return wide


print("[OK] DART wide 변환 모듈 로드 완료")


[OK] DART wide 변환 모듈 로드 완료


## Cell 3 · 계정 매핑 진단

In [9]:
# ═══════════════════════════════════════════════════════════════
#  진단 — 특정 종목의 DART 계정 목록 덤프 (매핑 보강용)
#  da=0 처럼 필드가 비면 이 셀을 실행해 실제 account_id/account_nm 을 확인하고
#  Cell 2 의 FIELD_MAP 에 패턴을 추가하세요.
# ═══════════════════════════════════════════════════════════════
import pandas as pd


def diagnose_accounts(ticker: str, sj_div: str = None, keyword: str = None,
                      top: int = 60):
    """
    종목의 (sj_div, account_id, account_nm) 별 커버리지·최근 FY 금액 요약.
    sj_div : 'CF','BS','IS','CIS' 필터 (None=전체)
    keyword: account_nm 포함 검색어 (예: '상각', '취득', '차입')
    """
    raw = _load_ticker_long(ticker, DB_INFO, TABLE_DART_FS)
    if raw.empty:
        print(f"❌ {ticker}: 데이터 없음")
        return None
    df = raw.copy()
    if sj_div:
        df = df[df["sj_div"] == sj_div]
    if keyword:
        df = df[df["account_nm"].astype(str).str.contains(keyword, na=False)]

    last_fy = df[df["quarter"] == "FY"]["bsns_year"].max()
    fy_amt = (df[(df["quarter"] == "FY") & (df["bsns_year"] == last_fy)]
              .set_index(["sj_div", "account_id", "account_nm"])["thstrm_amount"])

    g = (df.groupby(["sj_div", "account_id", "account_nm"])
           .agg(n_periods=("bsns_year", "count"),
                yr_min=("bsns_year", "min"), yr_max=("bsns_year", "max"))
           .sort_values("n_periods", ascending=False))
    g["last_FY_억원"] = (fy_amt.reindex(g.index) / 1e8).round(0)
    print(f"[{norm_ticker(ticker)}] sj={sj_div or 'ALL'} kw={keyword or '-'} "
          f"— 계정 {len(g)}개 (최근 FY={last_fy})")
    with pd.option_context("display.max_rows", top, "display.width", 200,
                           "display.max_colwidth", 45):
        print(g.head(top).to_string())
    return g


def check_field_mapping(ticker: str):
    """FIELD_MAP 각 필드가 이 종목에서 어떤 계정으로 resolve 되는지 일람."""
    raw = _load_ticker_long(ticker, DB_INFO, TABLE_DART_FS)
    print(f"[{norm_ticker(ticker)}] 필드 → 매칭 계정")
    for f, spec in FIELD_MAP.items():
        sel_rows = raw[raw["sj_div"].isin(spec["sj"])]
        hit = sel_rows[sel_rows["account_id"].isin(spec["ids"])] if spec["ids"] else sel_rows.iloc[0:0]
        via = "id"
        if hit.empty and spec["nm"]:
            for pat in spec["nm"]:
                m = sel_rows[sel_rows["account_nm"].astype(str).str.strip()
                             .str.contains(pat, regex=True, na=False)]
                if not m.empty:
                    hit, via = m, f"nm:{pat}"
                    break
        if hit.empty:
            print(f"  ❌ {f:<22} 매칭 없음")
        else:
            names = hit.groupby(["account_id", "account_nm"]).size() \
                       .sort_values(ascending=False)
            best = names.index[0]
            print(f"  ✅ {f:<22} [{via}] {best[0]} / {best[1]} "
                  f"({names.iloc[0]}기간{', 후보 '+str(len(names))+'개' if len(names)>1 else ''})")


# 실행 예: D&A가 0으로 나온 종목의 CF 계정 확인
check_field_mapping(CHECK_TICKERS[0])
print()
diagnose_accounts(CHECK_TICKERS[0], sj_div="CF", keyword="상각")


C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


[005930] 필드 → 매칭 계정
  ✅ revenue                [id] ifrs-full_Revenue / 수익(매출액) (17기간, 후보 4개)
  ✅ operating_income       [id] dart_OperatingIncomeLoss / 영업이익 (26기간, 후보 2개)
  ✅ pretax_income          [id] ifrs-full_ProfitLossBeforeTax / 법인세비용차감전순이익(손실) (20기간, 후보 3개)
  ✅ tax_expense            [id] ifrs-full_IncomeTaxExpenseContinuingOperations / 법인세비용 (21기간, 후보 3개)
  ✅ interest_expense       [id] ifrs-full_FinanceCosts / 금융비용 (12기간, 후보 2개)
  ❌ da_cf                  매칭 없음
  ❌ intangible_amort_cf    매칭 없음
  ✅ capex_tangible         [id] ifrs-full_PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities / 유형자산의 취득 (28기간, 후보 2개)
  ✅ capex_intangible       [id] ifrs-full_PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities / 무형자산의 취득 (28기간, 후보 2개)
  ✅ receivables            [id] ifrs-full_CurrentTradeReceivables / 매출채권 (11기간)
  ✅ inventories            [id] ifrs-full_Inventories / 재고자산 (28기간, 후보 2개)
  ✅ prepaid_expenses       [nm:^선급비용] ifrs-full_CurrentPrepaidExpenses / 선급비용

,,,n_periods,yr_min,yr_max,last_FY_억원
sj_div,account_id,account_nm,,,,


## Cell 4 · 품질 점검 실행

In [10]:
# ═══════════════════════════════════════════════════════════════
#  데이터 품질 검증 — 수집·분기화가 제대로 됐는지 종목별/일괄 점검
#  ① 필드 매핑 성공 여부   ② 누적/3개월 감지 결과
#  ③ 분기화 정합성 (연도별 Q1~Q4 + FY 원본 대조, 음수 분기 탐지)
#  ④ 최근 8분기 주요 항목 눈검사 테이블
# ═══════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd

KEY_FLOW_FIELDS  = ["revenue", "operating_income", "pretax_income",
                    "tax_expense", "da_cf", "capex_tangible"]
KEY_STOCK_FIELDS = ["receivables", "inventories", "payables",
                    "total_equity", "total_assets", "cash"]
CHECK_FIELDS = KEY_FLOW_FIELDS + KEY_STOCK_FIELDS


def _quarterize_report(ticker: str, field: str = "revenue"):
    """
    flow 필드 1개의 분기화 과정을 연도별 표로 보여준다.
    원본 (Q1,H1,Q3,FY raw) + 감지 모드 + 분기화 결과 (q1~q4) + 정합성 플래그
    """
    raw = _load_ticker_long(ticker, DB_INFO, TABLE_DART_FS)
    spec = FIELD_MAP[field]
    sel = _match_field(raw, spec)
    if sel.empty:
        print(f"  ({field}: 매칭 계정 없음)")
        return None

    pivot = sel.pivot_table(index="bsns_year", columns="quarter",
                            values="thstrm_amount", aggfunc="first")
    force = True if spec["sj"] == ["CF"] else None
    qvals, cum = _flow_to_quarterly(sel, force_cumulative=force)

    rows = []
    for y in sorted(pivot.index):
        r = {"year": int(y)}
        for q in ("Q1", "H1", "Q3", "FY"):
            r[f"raw_{q}"] = pivot.loc[y].get(q, np.nan)
        for q in ("Q1", "Q2", "Q3", "Q4"):
            r[q.lower()] = qvals.get((y, q), np.nan)
        qs = [r["q1"], r["q2"], r["q3"], r["q4"]]
        r["q_sum"] = np.nansum(qs) if any(pd.notnull(v) for v in qs) else np.nan
        fy = r["raw_FY"]
        r["neg_q"] = int(sum(1 for v in qs if pd.notnull(v) and v < 0))
        r["sum≈FY"] = ("OK" if pd.notnull(fy) and pd.notnull(r["q_sum"])
                       and abs(r["q_sum"] - fy) <= abs(fy) * 0.001 else
                       ("-" if pd.isnull(fy) else "MISMATCH"))
        rows.append(r)

    df = pd.DataFrame(rows).set_index("year")
    print(f"  [{field}] 감지 모드: {'누적(차분 적용)' if cum else '3개월치(Q4=FY−3분기합)'}")
    num_cols = [c for c in df.columns if c not in ("neg_q", "sum≈FY")]
    disp = (df[num_cols] / 1e8).round(0)   # 억원
    disp["neg_q"] = df["neg_q"]
    disp["sum≈FY"] = df["sum≈FY"]
    print(disp.to_string())
    n_neg = int(df["neg_q"].sum())
    n_mis = int((df["sum≈FY"] == "MISMATCH").sum())
    if n_neg:
        print(f"  ⚠️ 음수 분기 {n_neg}개 — 누적/3개월 오감지 또는 정정공시 혼입 의심")
    if n_mis:
        print(f"  ⚠️ 분기합≠FY 연도 {n_mis}개")
    return dict(cumulative=cum, neg_quarters=n_neg, fy_mismatch=n_mis)


def check_ticker(ticker: str, show_quarterize=("revenue", "operating_income", "da_cf")):
    """단일 종목 종합 점검 리포트."""
    tk = norm_ticker(ticker)
    print("=" * 78)
    print(f"■ {tk} 데이터 품질 점검")
    print("=" * 78)

    wide = load_dart_financials_wide(tk, DB_INFO, TABLE_DART_FS, verbose=True)
    if wide.empty:
        print("❌ 데이터 없음")
        return {"ticker": tk, "status": "NO_DATA"}

    # ① 필드 커버리지
    print("\n[① 핵심 필드 커버리지] (비결측 분기 수 / 전체", len(wide), "분기)")
    stat = {}
    for f in CHECK_FIELDS:
        n = int(wide[f].notna().sum()) if f in wide.columns else 0
        stat[f] = n
        mark = "✅" if n >= 12 else ("⚠️" if n > 0 else "❌")
        print(f"  {mark} {f:<20} {n}")

    # ②③ 분기화 상세 (지정 flow 필드)
    print("\n[② 분기화 정합성] (단위: 억원)")
    qz = {}
    for f in show_quarterize:
        res = _quarterize_report(tk, f)
        if res:
            qz[f] = res
        print()

    # ④ 최근 8분기 눈검사
    print("[③ 최근 8분기 주요 항목 (조원)]")
    key = [c for c in CHECK_FIELDS if c in wide.columns]
    print((wide[key].tail(8) / 1e12).round(3).to_string())

    return {"ticker": tk, "status": "OK",
            "n_quarters": len(wide),
            **{f"n_{f}": stat.get(f, 0) for f in CHECK_FIELDS},
            "rev_neg_q": qz.get("revenue", {}).get("neg_quarters", np.nan),
            "rev_cumulative_detected": qz.get("revenue", {}).get("cumulative", None)}


def check_batch(tickers):
    """여러 종목 요약 매트릭스 — '수집이 잘 되는가'를 한 눈에."""
    rows = []
    for tk in tickers:
        try:
            rows.append(check_ticker(tk))
        except Exception as e:
            print(f"❌ {tk}: {e}")
            rows.append({"ticker": norm_ticker(tk), "status": f"ERROR: {str(e)[:60]}"})
        print()
    summary = pd.DataFrame(rows)
    print("=" * 78)
    print("■ 일괄 점검 요약")
    print("=" * 78)
    with pd.option_context("display.width", 220, "display.max_columns", 30):
        print(summary.to_string(index=False))
    return summary


# ── 실행 ──
batch_summary = check_batch(CHECK_TICKERS)


■ 005930 데이터 품질 점검


C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


[11:18:51][005930] wide shape=(48, 24) 기간=2015-03-31~2026-12-31  누적차분 적용: ['capex_tangible', 'capex_intangible']

[① 핵심 필드 커버리지] (비결측 분기 수 / 전체 48 분기)
  ✅ revenue              42
  ✅ operating_income     42
  ✅ pretax_income        42
  ✅ tax_expense          42
  ❌ da_cf                0
  ✅ capex_tangible       42
  ✅ receivables          35
  ✅ inventories          43
  ✅ payables             12
  ✅ total_equity         31
  ✅ total_assets         43
  ✅ cash                 43

[② 분기화 정합성] (단위: 억원)
  [revenue] 감지 모드: 3개월치(Q4=FY−3분기합)
         raw_Q1     raw_H1    raw_Q3     raw_FY         q1         q2        q3        q4      q_sum  neg_q    sum≈FY
year                                                                                                                 
2015        NaN        NaN       NaN  2006535.0        NaN        NaN       NaN       NaN        NaN      0  MISMATCH
2016   497823.0   509371.0  478156.0  2018667.0   497823.0   509371.0  478156.0  533317.0  2018667.0  

C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (eng

  (da_cf: 매칭 계정 없음)

[③ 최근 8분기 주요 항목 (조원)]
            revenue  operating_income  pretax_income  tax_expense  capex_tangible  receivables  inventories  payables  total_equity  total_assets    cash
2025-03-31   79.141             6.685          9.152        0.929          12.128       44.867       53.220    14.496           NaN       516.377  53.161
2025-06-30   74.566             4.676          5.756        0.640          13.035       43.551       51.037    12.676           NaN       504.875  47.120
2025-09-30   86.062            12.166         13.546        1.320          10.810       50.539       50.332    14.417           NaN       523.660  53.399
2025-12-31   93.837            20.074         21.028        1.386          11.549       51.128       52.637    13.039           NaN       566.942  57.856
2026-03-31  133.873            57.233         58.828       11.603          17.127       82.285       58.278    15.821           NaN       633.340  73.307
2026-06-30  171.499            89

...

[11:18:53][005380] wide shape=(48, 22) 기간=2015-03-31~2026-12-31  누적차분 적용: ['capex_tangible', 'capex_intangible']

[① 핵심 필드 커버리지] (비결측 분기 수 / 전체 48 분기)
  ✅ revenue              42
  ✅ operating_income     33
  ✅ pretax_income        42
  ✅ tax_expense          42
  ❌ da_cf                0
  ✅ capex_tangible       42
  ✅ receivables          43
  ✅ inventories          43
  ✅ payables             43
  ✅ total_equity         31
  ✅ total_assets         43
  ✅ cash                 43

[② 분기화 정합성] (단위: 억원)
  [revenue] 감지 모드: 3개월치(Q4=FY−3분기합)
        raw_Q1    raw_H1    raw_Q3     raw_FY        q1        q2        q3        q4      q_sum  neg_q    sum≈FY
year                                                                                                             
2015       NaN       NaN       NaN   919587.0       NaN       NaN       NaN       NaN        NaN      0  MISMATCH
2016  223506.0  246767.0  220837.0   936490.0  223506.0  246767.0  220837.0  245380.0   936490.0      0        OK


C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (eng

  (da_cf: 매칭 계정 없음)

[③ 최근 8분기 주요 항목 (조원)]
            revenue  operating_income  pretax_income  tax_expense  capex_tangible  receivables  inventories  payables  total_equity  total_assets    cash
2025-03-31   44.408             3.634          4.465        1.082           2.085        5.996       20.715    13.191           NaN       343.630  17.998
2025-06-30   48.287             3.602          4.385        1.135           1.772        6.658       21.287    12.845           NaN       339.702  17.520
2025-09-30   46.721             2.537          3.326        0.778           1.689        7.089       21.193    12.147           NaN       354.432  17.861
2025-12-31   46.839             1.695          1.666        0.482           2.821        8.600       20.662    12.287           NaN       368.845  18.361
2026-03-31   45.939             2.515          3.522        0.937           2.521        6.162       22.572    13.071           NaN       383.834  18.977
2026-06-30   49.215             2

C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


  (da_cf: 매칭 계정 없음)

[③ 최근 8분기 주요 항목 (조원)]
            revenue  operating_income  pretax_income  tax_expense  capex_tangible  receivables  inventories  payables  total_equity  total_assets    cash
2025-03-31   12.171             0.447          0.444        0.162          -4.044        8.783        8.529     3.925           NaN        95.091   6.950
2025-06-30   11.418             0.477         -0.185       -0.068          -3.550        7.651        7.846     3.187           NaN        93.952   8.383
2025-09-30   11.196             0.680          0.457       -0.007          -3.086        7.710        8.629     3.478           NaN        98.475   8.590
2025-12-31   11.147            -0.423         -2.497       -0.071          24.340        6.725        8.177     3.537           NaN       101.062   9.900
2026-03-31   12.247            -0.050         -0.566        0.215           2.240        8.343        9.351     4.513           NaN       105.663   8.532
2026-06-30   14.176             0

## Cell 5 · D&A proxy 검증

삼성전자(da_cf 실측 보유)에서 proxy/actual 비율 중앙값이 **0.8~1.2** 안에 들어오면
SK하이닉스처럼 CF에 D&A가 없는 기업에도 proxy를 안심하고 적용할 수 있습니다.

In [11]:
# ═══════════════════════════════════════════════════════════════
#  D&A proxy — CF에 감가상각비가 없는 기업(SK하이닉스 등) 대응
#    D&A_q ≈ CapEx_q − Δ(유형자산 + 무형자산 장부가)
#  da_cf 실측이 있는 기업(삼성전자 등)으로 proxy 정확도를 먼저 검증
# ═══════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd


def estimate_da_proxy(ticker: str, show: bool = True):
    """
    반환: DataFrame[da_est, (da_cf_actual, ratio)]
    ratio 중앙값이 0.8~1.2 면 proxy 신뢰 가능.
    """
    tk = norm_ticker(ticker)
    w = load_dart_financials_wide(tk, DB_INFO, TABLE_DART_FS)
    if w.empty or "ppe" not in w.columns or "capex_tangible" not in w.columns:
        missing = {"ppe", "capex_tangible"} - set(w.columns)
        print(f"❌ {tk}: proxy 산출 불가 — 필드 부족 {missing}")
        return None

    capex = w["capex_tangible"].abs()
    if "capex_intangible" in w.columns:
        capex = capex.add(w["capex_intangible"].abs(), fill_value=0)

    book = w["ppe"].astype(float)
    if "intangible_assets" in w.columns:
        book = book.add(w["intangible_assets"].astype(float), fill_value=0)

    # D&A ≈ CapEx − ΔBook (처분·손상·재평가 잡음 → 음수는 0 클립)
    da_est = (capex - book.diff()).clip(lower=0)
    out = pd.DataFrame({"da_est": da_est})

    verdict = None
    if "da_cf" in w.columns and w["da_cf"].notna().sum() >= 8:
        out["da_cf_actual"] = w["da_cf"]
        out["ratio"] = (out["da_est"] / out["da_cf_actual"]) \
            .replace([np.inf, -np.inf], np.nan)
        med = float(out["ratio"].dropna().median())
        verdict = "✅ proxy 신뢰 가능" if 0.8 <= med <= 1.2 else "⚠️ 괴리 큼 — 원인 점검 필요"
        print(f"[{tk}] proxy/actual 비율 중앙값 = {med:.2f}  {verdict}")
    else:
        print(f"[{tk}] da_cf 실측 없음 → proxy만 산출 (실측 보유 종목으로 교차검증 권장)")

    if show:
        print((out.tail(10) / 1e12 if "ratio" not in out.columns
               else out.assign(**{c: out[c] / 1e12 for c in out.columns if c != "ratio"})
                      .tail(10)).round(3).to_string())
    return out


# ── 검증 실행 ──
# 1) 실측 보유 종목으로 proxy 정확도 확인 (비율 중앙값 0.8~1.2 목표)
estimate_da_proxy("005930")
print()
# 2) CF에 D&A 없는 종목 — proxy 산출
estimate_da_proxy("000660")


C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


[005930] da_cf 실측 없음 → proxy만 산출 (실측 보유 종목으로 교차검증 권장)
            da_est
2024-09-30  11.220
2024-12-31   4.166
2025-03-31   8.988
2025-06-30  16.875
2025-09-30  11.884
2025-12-31   0.000
2026-03-31  15.504
2026-06-30   8.484
2026-09-30     NaN
2026-12-31     NaN



C:\Users\82108\AppData\Local\Temp\ipykernel_17344\3812802823.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


[000660] da_cf 실측 없음 → proxy만 산출 (실측 보유 종목으로 교차검증 권장)
            da_est
2024-09-30   2.286
2024-12-31   1.848
2025-03-31   3.653
2025-06-30   3.181
2025-09-30   1.576
2025-12-31   2.793
2026-03-31   3.315
2026-06-30   4.059
2026-09-30     NaN
2026-12-31     NaN


,da_est
2015-03-31,NaN
2015-06-30,NaN
2015-09-30,NaN
2015-12-31,NaN
2016-03-31,1.048306e+12
2016-06-30,1.427304e+12
2016-09-30,1.477630e+12
2016-12-31,5.116440e+11
2017-03-31,1.047137e+12
2017-06-30,1.200984e+12
